# AgentCore Memory Branching으로 병렬 실행하는 Strands Multi-Agent System

## 소개

이 Notebook에서는 여러 전문 Agent가 공통 Memory 리소스를 공유하면서 격리된 Memory 컨텍스트를 유지하도록 하는 강력한 기능인 **AgentCore Memory Branching**을 살펴봅니다. 이는 Multi-Agent System, 특히 Strands Agent Graph처럼 병렬 실행 pattern을 사용하는 시스템에 필수적입니다.

## Memory Branching이 중요한 이유

Multi-Agent System에서는 서로 다른 Agent에 다음 기능이 필요한 경우가 많습니다.
- **분리된 대화 컨텍스트 유지** - 각 Agent가 간섭 없이 담당 도메인에 집중
- **병렬 실행** - 여러 Agent가 Memory 충돌 없이 동시에 작업
- **공통 세션 공유** - 모든 Agent가 컨텍스트를 격리한 채 동일한 사용자 세션에 기여
- **관련 기록 액세스** - Agent가 컨텍스트 혼합 없이 자체 과거 상호작용 검색

AgentCore Memory Branching은 코드의 Git branch와 마찬가지로 하나의 Memory 세션 안에 여러 대화 branch를 허용하여 이러한 문제를 해결합니다.

## 튜토리얼 세부 정보

| 정보         | 세부 정보                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | Memory Branching을 사용하는 Multi-Agent                                                |
| Agent 사용 사례       | Travel Planning Assistant                                                        |
| Agentic Framework   | Strands Agent Graph (supports parallel execution)                                |
| LLM 모델           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소 | AgentCore Memory Branching, Strands Multi-Agent Graph, 병렬 실행        |
| 예제 난이도  | 중급                                                                     |


학습 내용:

- 서로 다른 Agent의 Memory branch를 생성하고 관리하는 방법
- Multi-Agent 아키텍처에서 격리된 Memory 컨텍스트 구현
- 병렬 실행을 지원하는 Strands Agent Graph 구축
- Branching으로 안전한 동시 Memory 액세스를 지원하는 방법
- Branch별 대화 기록 확인 및 살펴보기

### 시나리오 배경

각각 자체 Memory branch를 사용하는 세 Agent로 **여행 계획 시스템**을 구축합니다.
1. **Travel Coordinator**(main branch) - 전체 여행 계획 조율
2. **Flight Booking Assistant**(flight_agent_memory branch) - 항공 여행 질의 처리
3. **Hotel Booking Assistant**(hotel_agent_memory branch) - 숙박 요청 관리

Coordinator는 병렬로 실행되는 전문 Agent에 작업을 위임할 수 있으며, 각 Agent는 Memory Branching을 통해 자체 대화 기록을 유지합니다.

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## 사전 요구 사항
- Python 3.10 이상
- 적절한 권한이 있는 AWS 계정
- AgentCore Memory에 적절한 권한이 있는 AWS IAM 역할
- Amazon Bedrock 모델에 대한 액세스

먼저 환경을 설정하고 Branching을 지원하는 공유 Memory 리소스를 생성하겠습니다.

## 1단계: 환경 설정
Notebook 실행에 필요한 모든 라이브러리를 가져오고 client를 정의합니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
from datetime import datetime
from strands.hooks import (
    AgentInitializedEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

Amazon Bedrock 모델 및 AgentCore에 적절한 권한이 있는 리전과 역할을 정의합니다.

In [ ]:
import os

region = os.getenv("AWS_REGION", "us-west-2")
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("agentcore-memory")

## 2단계: Branching을 지원하는 공유 Memory 생성

각 Agent마다 하나씩 여러 branch를 지원하는 단일 Memory 리소스를 생성합니다. 이 공유 Memory 리소스는 기반 역할을 하며, branch는 각 Agent의 대화에 격리된 컨텍스트를 제공합니다.

여러 branch(Agent 컨텍스트)를 가진 하나의 Git repository(Memory 리소스)와 비슷합니다.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

In [ ]:
client = MemoryClient(region_name=region)
memory_name = "TravelAgent_STM_%s" % datetime.now().strftime("%Y%m%d%H%M%S")
memory_id = None

In [ ]:
from botocore.exceptions import ClientError

try:
    print("Creating Memory...")
    memory_name = memory_name

    # Memory 리소스 생성
    memory = client.create_memory_and_wait(
        name=memory_name,  # 이 Memory 저장소의 고유 이름
        description="Travel Agent STM",  # 사람이 읽을 수 있는 설명
        strategies=[],  # 단기 메모리에는 별도 Memory strategy를 사용하지 않음
        event_expiry_days=7,  # Memory는 7일 후 만료
        max_wait=300,  # Memory 생성을 기다리는 최대 시간(5분)
        poll_interval=10,  # 10초마다 상태 확인
    )

    # Memory ID 추출 및 출력
    memory_id = memory["id"]
    print(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(e):
        # Memory가 이미 존재하면 ID 검색
        memories = client.list_memories()
        memory_id = next((m["id"] for m in memories if m["id"].startswith(memory_name)), None)
        logger.info(f"Memory already exists. Using existing memory ID: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생한 오류 처리
    print(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()

    # 오류 발생 시 정리 - 일부 생성된 Memory 삭제
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### Multi-Agent System의 Memory Branching 이해

생성한 Memory 리소스는 Multi-Agent 아키텍처의 핵심 기능인 **Branching**을 지원합니다. 작동 방식은 다음과 같습니다.

**단일 Memory 리소스, 여러 Branch:**
- 모든 Agent가 동일한 `memory_id`와 `session_id` 공유
- 각 Agent가 격리된 컨텍스트를 위한 자체 `branch_name` 사용

**Multi-Agent System의 주요 이점:**

1. **컨텍스트 격리**: 각 Agent가 간섭 없이 자체 대화 기록 유지
- Flight Agent는 항공편 관련 대화만 확인
- Hotel Agent는 호텔 관련 대화만 확인
- Coordinator는 주요 조율 흐름 확인

2. **병렬 실행 안전성**: 여러 Agent를 동시에 실행 가능
- Agent 병렬 실행 시 Memory 충돌 없음
- 각 branch에 독립적으로 액세스 가능
- 동시 실행을 지원하는 Strands Agent Graph에 필수

3. **명확한 Audit Trail**: 각 Agent의 상호작용 추적 가능
- 각 Agent가 논의한 내용 확인
- Agent별 문제 debugging
- Multi-Agent 대화 흐름 이해

## 3단계: Branch를 지원하는 Memory Hook Provider 생성

`ShortTermMemoryHook` 클래스는 branch를 인식하는 메모리 관리를 구현합니다. 이는 Multi-Agent System에서 Memory Branching을 지원하는 핵심 구성 요소입니다.

**주요 기능:**

1. **Branch 초기화**: 각 Agent의 branch를 자동 생성
- Coordinator Agent용 main branch
- Sub-Agent용 전문 branch(예: `flight_agent_memory`, `hotel_agent_memory`)
- Main 대화 timeline에서 branch fork

2. **Branch별 Memory 검색**: 각 Agent가 자체 컨텍스트만 불러옴
- `on_agent_initialized()`가 Agent branch에서 대화 기록 가져오기
- Agent 간 컨텍스트 오염 방지
- Agent가 집중된 도메인별 대화를 유지하도록 지원

3. **Branch별 Memory 저장**: 대화를 올바른 branch에 저장
- `on_message_added()`가 Agent의 지정된 branch에 메시지 저장
- Agent 병렬 실행의 동시 쓰기 지원
- race condition 또는 Memory 충돌 없음

이 Hook Provider는 AgentCore Memory를 사용하는 Agent 병렬 실행을 안전하고 효율적으로 만듭니다.

In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from bedrock_agentcore.memory import MemorySessionManager


class ShortTermMemoryHook(HookProvider):
    def __init__(self, memory_id: str, region_name: str = "us-west-2", branch_name: str = "main"):
        """MemorySessionManager로 훅을 초기화합니다.

        인자:
            memory_id: AgentCore Memory ID
            region_name: 메모리 서비스용 AWS 리전
            branch_name: 이 에이전트 메모리의 브랜치 이름(기본값: "main")
        """
        self.memory_manager = MemorySessionManager(memory_id=memory_id, region_name=region_name)
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # actor/session 조합별 세션 객체 cache
        self._branch_initialized = False  # Branch 생성 여부 추적

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """주어진 행위자와 세션의 MemorySession을 가져오거나 생성합니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자

        반환값:
            MemorySession 객체
        """
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(actor_id=actor_id, session_id=session_id)
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """main 브랜치가 아니면서 브랜치가 없으면 초기화합니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자
        """
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Branch가 이미 존재하는지 확인
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # Fork할 main branch의 마지막 event 가져오기
                main_events = memory_session.list_events(branch_name="main")
                if main_events:
                    last_event = main_events[-1]
                    # 초기 메시지와 함께 branch 생성
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(
                                f"Starting {self.branch_name} branch",
                                MessageRole.ASSISTANT,
                            )
                        ],
                    )
                    logger.info(f"✅ Created branch: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(f"Failed to initialize branch {self.branch_name}: {e}", exc_info=True)

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """에이전트가 시작될 때 최근 대화 기록을 불러옵니다."""
        try:
            # Agent 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # Memory Session 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # main branch에 event가 있으면 main이 아닌 branch 초기화
            if self.branch_name != "main":
                try:
                    main_events = memory_session.list_events(branch_name="main")
                    if len(main_events) > 0:
                        self._initialize_branch(actor_id, session_id)
                except Exception as e:
                    # 첫 호출에서는 main branch가 아직 없을 수 있음
                    logger.info(f"Main branch not found yet, will initialize {self.branch_name} branch later: {e}")

            # Turn을 가져오기 전에 branch 존재 여부 확인
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            recent_turns = []
            if branch_exists:
                # Branch가 있는 경우에만 turn 가져오기
                recent_turns = memory_session.get_last_k_turns(k=5, branch_name=self.branch_name)
            else:
                logger.info(f"Branch '{self.branch_name}' does not exist yet, skipping turn retrieval")

            if len(recent_turns) > 0:
                # 대화 기록을 컨텍스트 형식으로 변환
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.get("role", "unknown").lower()
                        text = message.get("content", {}).get("text", "")
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages)
                    logger.info(f"Loaded context from branch '{self.branch_name}' ({len(context_messages)} messages)")

                    # Agent의 system prompt에 컨텍스트 추가
                    event.agent.system_prompt += (
                        f"\n\nRecent conversation history (from {self.branch_name}):\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )

                    logger.info(
                        f"✅ Loaded {len(recent_turns)} recent conversation turns from branch '{self.branch_name}'"
                    )
            else:
                logger.info(f"No previous conversation history found in branch '{self.branch_name}'")

        except Exception as e:
            logger.error(f"Failed to load conversation history: {e}", exc_info=True)

    def on_message_added(self, event: MessageAddedEvent):
        """대화 턴을 메모리의 적절한 브랜치에 저장합니다."""
        try:
            # Agent 상태에서 세션 정보 가져오기
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning("Missing actor_id or session_id in agent state")
                return

            # Memory Session 가져오기
            memory_session = self._get_or_create_session(actor_id, session_id)

            # 마지막 메시지 가져오기
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("Skipping empty message")
                return

            # 역할 문자열을 MessageRole enum에 매핑
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # 메시지를 적절한 branch에 저장
            if self.branch_name == "main":
                # Main branch - 일반적인 방식으로 turn 추가
                memory_session.add_turns(messages=[ConversationalMessage(content_text, message_role)])
            else:
                # Main이 아닌 branch - 기존 branch에 추가해야 함
                # Branch가 없으면 초기화
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # 이 branch의 최신 event 가져오기
                branch_events = memory_session.list_events(branch_name=self.branch_name)
                if branch_events:
                    # Branch 이름을 지정하여 기존 branch에 추가(rootEventId 제외)
                    memory_session.add_turns(
                        messages=[ConversationalMessage(content_text, message_role)],
                        branch={"name": self.branch_name},
                    )
                else:
                    # _initialize_branch가 작동했다면 발생하지 않아야 하지만 예외적으로 처리
                    logger.warning(f"Branch {self.branch_name} not found after initialization")
                    self._initialize_branch(actor_id, session_id)

            logger.debug(f"✅ Stored message in branch '{self.branch_name}': {role_str}")

        except Exception as e:
            logger.error(f"Failed to store message: {e}", exc_info=True)

    def create_branch(
        self,
        actor_id: str,
        session_id: str,
        root_event_id: str,
        branch_name: str,
        messages: list,
    ):
        """새 대화 브랜치를 생성합니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자
            root_event_id: 분기 기준 이벤트 ID
            branch_name: 새 브랜치 이름
            messages: 브랜치에 추가할 ConversationalMessage 객체 목록
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.fork_conversation(root_event_id=root_event_id, branch_name=branch_name, messages=messages)

    def list_branches(self, actor_id: str, session_id: str):
        """세션의 모든 브랜치를 나열합니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자

        반환값:
            브랜치 정보 목록
        """
        memory_session = self._get_or_create_session(actor_id, session_id)
        return memory_session.list_branches()

    def get_session(self, actor_id: str, session_id: str):
        """직접 액세스할 메모리 세션 객체를 가져옵니다.

        인자:
            actor_id: 행위자 식별자
            session_id: 세션 식별자

        반환값:
            MemorySession 객체
        """
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """메모리 훅을 레지스트리에 등록합니다.

        인자:
            registry: 콜백을 등록할 HookRegistry
        """
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

## 4단계: Strands Agent Graph로 Multi-Agent 아키텍처 생성

이제 Agent 병렬 실행을 지원하는 framework인 **Strands Agent Graph**로 Multi-Agent System을 구축합니다. 안전한 동시 작업을 위해 각 Agent에 자체 Memory branch를 구성합니다.

**아키텍처 개요:**
- **Coordinator Agent** → `main` branch 사용
- **Flight Agent** → `flight_agent_memory` branch 사용
- **Hotel Agent** → `hotel_agent_memory` branch 사용

모든 Agent가 동일한 `session_id`를 공유하지만 Branching을 통해 격리된 대화 컨텍스트를 유지합니다.

In [ ]:
# 필요한 구성 요소 가져오기
from strands import Agent

In [ ]:
# 각 전문 Agent에 고유 Actor ID를 생성하되 Session ID는 공유
actor_id = f"travel-user-{datetime.now().strftime('%Y%m%d%H%M%S')}"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"
namespace = f"travel/{actor_id}/preferences/"

### Branch별 Memory를 사용하는 전문 Agent 생성

전문 Agent의 system prompt를 정의합니다. 각 Agent에 자체 Memory branch를 구성하여 대화 격리를 보장하고 병렬 실행을 지원합니다.

In [ ]:
# Hotel Booking 전문 Agent용 system prompt
HOTEL_BOOKING_PROMPT = """You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities. 
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner."""

# Flight Booking 전문 Agent용 system prompt
FLIGHT_BOOKING_PROMPT = """You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies. 
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner."""

In [ ]:
flight_memory_hooks = None
hotel_memory_hooks = None

### Memory Branching을 사용하는 Agent 구현

각 전문 Agent에는 다음이 구성됩니다.
- 격리된 Memory 컨텍스트를 위한 고유 `branch_name`
- 공유 세션 관리를 위한 동일한 `memory_id` 및 `session_id`
- Branch별 작업을 처리하는 `ShortTermMemoryHook`

**주요 구현 세부 정보:**
- `flight_booking_agent()`는 `flight_agent_memory` branch 사용
- `hotel_booking_agent()`는 `hotel_agent_memory` branch 사용

In [ ]:
def flight_booking_agent() -> Agent:
    global flight_memory_hooks
    try:
        if flight_memory_hooks is None:
            # "flight_agent_memory" 이름의 branch를 사용하는 Hook 생성
            flight_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="flight_agent_memory",
            )

        flight_agent = Agent(
            hooks=[flight_memory_hooks],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id},
        )

        return flight_agent
    except Exception as e:
        return f"Error in flight booking assistant: {str(e)}"


def hotel_booking_agent() -> Agent:
    global hotel_memory_hooks
    try:
        if hotel_memory_hooks is None:
            # "hotel_agent_memory" 이름의 branch를 사용하는 Hook 생성
            hotel_memory_hooks = ShortTermMemoryHook(
                memory_id=memory_id,
                region_name=region,
                branch_name="hotel_agent_memory",
            )

        hotel_booking_agent = Agent(
            hooks=[hotel_memory_hooks],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT,
            state={"actor_id": actor_id, "session_id": session_id},
        )

        return hotel_booking_agent
    except Exception as e:
        return f"Error in hotel booking assistant: {str(e)}"

### Coordinator Agent 생성

Coordinator Agent는 `main` branch(기본값)를 사용하여 전문 Agent를 조율합니다. Strands Agent Graph를 사용할 때 병렬로 실행될 수 있는 Flight 및 Hotel Agent에 작업을 위임할 수 있습니다.

In [ ]:
# Coordinator Agent용 system prompt
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_agent
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_agent
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources. \
Ask max two questions per turn. Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
def travel_booking_agent() -> Agent:
    agent_memory_hooks = ShortTermMemoryHook(
        memory_id=memory_id,
        region_name=region,
    )
    travel_agent = Agent(
        system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
        hooks=[agent_memory_hooks],
        model=MODEL_ID,
        state={"actor_id": actor_id, "session_id": session_id},
    )

    return travel_agent

### 병렬 실행을 지원하는 Agent Graph 구축

이제 Agent를 **Strands Agent Graph**로 구성합니다. 이 Graph 구조는 다음을 지원합니다.

**병렬 실행:**
- Coordinator에 항공편과 호텔 정보가 모두 필요하면 두 Agent를 동시에 실행
- Memory Branching으로 동시 실행 중 충돌 방지
- 각 Agent가 자체 branch를 독립적으로 읽고 쓰기

**Memory Branch 매핑:**
```
Session: travel-session-xxx
├── main branch              → Travel Coordinator
├── flight_agent_memory      → Flight Booking Agent
└── hotel_agent_memory       → Hotel Booking Agent
```

**중요한 이유:**
- Branching이 없으면 병렬 Agent가 서로의 Memory를 덮어씀
- Branching을 사용하면 각 Agent가 자체 대화 thread 유지
- Coordinator가 여러 Agent에 동시에 안전하게 작업 위임

In [ ]:
import logging
from strands import Agent
from strands.multiagent import GraphBuilder

# debug log를 활성화하고 stderr로 출력
logging.getLogger("strands.multiagent").setLevel(logging.DEBUG)
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

# Strands Agent Graph 구축
# 이 Graph 구조로 전문 Agent 병렬 실행 지원
# Memory Branching으로 충돌 없는 안전한 동시 액세스 보장
builder = GraphBuilder()

# Node 추가 - 각 Agent가 자체 Memory branch 사용
builder.add_node(travel_booking_agent(), "travel_agent")  # 'main' branch 사용
builder.add_node(flight_booking_agent(), "flight_booking_agent")  # 'flight_agent_memory' branch 사용
builder.add_node(hotel_booking_agent(), "hotel_booking_agent")  # 'hotel_agent_memory' branch 사용

# Edge 추가 - Coordinator가 작업을 위임할 수 있는 Agent 정의
# 두 Agent가 모두 필요하면 Graph가 Flight 및 Hotel Agent를 병렬 실행
builder.add_edge("travel_agent", "flight_booking_agent")
builder.add_edge("travel_agent", "hotel_booking_agent")

# Entry point 설정 - Coordinator Agent가 사용자 입력을 먼저 수신
builder.set_entry_point("travel_agent")

# 안전을 위한 실행 제한 구성
builder.set_execution_timeout(600)  # 10분 timeout

# Graph build - 격리된 Memory 컨텍스트로 병렬 실행 준비
graph = builder.build()

### Memory Branching을 사용하는 Multi-Agent System 준비 완료

Agent Graph에 다음이 구성되었습니다.
- ✅ 격리된 Memory branch를 사용하는 세 Agent
- ✅ Strands Agent Graph를 통한 병렬 실행 기능
- ✅ AgentCore Memory Branching을 통한 안전한 동시 Memory 액세스
- ✅ Branch 자동 생성 및 관리

## Multi-Agent System 테스트

여러 Agent를 trigger하는 여행 계획 시나리오로 테스트해 보겠습니다.

In [ ]:
response = graph("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")

## Memory Branch 살펴보기

AgentCore Memory Branching의 주요 이점 중 하나는 각 Agent의 대화 기록을 독립적으로 살펴볼 수 있다는 점입니다. 이는 다음 작업에 중요합니다.

**Multi-Agent System debugging:**
- 각 Agent가 논의한 내용을 정확히 확인
- 대화의 각 부분을 처리한 Agent 식별
- 시스템 내 정보 흐름 추적

**병렬 실행 이해:**
- Agent가 분리된 컨텍스트를 유지했는지 검증
- 동시 실행 중 Memory 충돌이 발생하지 않았는지 확인
- Agent 상호작용 timeline 감사

대화 중 생성된 branch를 살펴보겠습니다.

In [ ]:
print("\n=== Viewing Memory Branches ===")

if flight_memory_hooks or hotel_memory_hooks:
    # Branch를 나열할 임의의 Memory Session 가져오기(모두 동일한 세션을 가리킴)
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # 세션의 모든 branch 나열
        branches = memory_session.list_branches()
        print(f"\n📊 Session has {len(branches)} branches total:")
        for branch in branches:
            print(f"  - Branch: {branch.name}")
            print(f"    └─ Events: {len(memory_session.list_events(branch_name=branch.name))}")
            print(f"    └─ Created: {branch.created}")

        print("\n💡 Each branch represents a different agent's memory:")
        print("  • 'main' = Travel coordinator conversations")
        print("  • 'flight_agent_memory' = Flight assistant conversations")
        print("  • 'hotel_agent_memory' = Hotel assistant conversations")

### Branch별 대화 기록 액세스

이제 각 branch에 저장된 실제 대화를 자세히 살펴봅니다. 이를 통해 Memory Branching이 공유 세션을 유지하면서 Agent 간에 완전한 격리를 제공하는 방법을 확인할 수 있습니다.

In [ ]:
print("\n=== Accessing Branch-Specific Events ===")

if flight_memory_hooks or hotel_memory_hooks:
    hook = flight_memory_hooks if flight_memory_hooks else hotel_memory_hooks
    if hook:
        memory_session = hook.get_session(actor_id, session_id)

        # Main branch(Coordinator)에서 event 가져오기
        main_events = memory_session.list_events(branch_name="main")
        print(f"\n🌳 Main Branch - Coordinator ({len(main_events)} events):")
        if main_events:
            for event in main_events[-3:]:  # 최근 event 3개 표시
                for payload in event.payload:
                    if "conversational" in payload:
                        role = payload["conversational"]["role"]
                        text = payload["conversational"]["content"]["text"]
                        print(f"  {role}: {text[:100]}...")
        else:
            print("  No events found in main branch")

        # Flight Agent branch에서 event 가져오기
        try:
            flight_branch_events = memory_session.list_events(branch_name="flight_agent_memory")
            print(f"\n✈️  Flight Agent Branch ({len(flight_branch_events)} events):")
            if flight_branch_events:
                print("All flight-related conversations are stored here:")
                for event in flight_branch_events[-3:]:  # 최근 event 3개 표시
                    for payload in event.payload:
                        if "conversational" in payload:
                            role = payload["conversational"]["role"]
                            text = payload["conversational"]["content"]["text"]
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  No events found - flight assistant wasn't called yet")
        except Exception as e:
            print(f"  Flight branch not created yet: {e}")

        # Hotel Agent branch에서 event 가져오기
        try:
            hotel_branch_events = memory_session.list_events(branch_name="hotel_agent_memory")
            print(f"\n🏨 Hotel Agent Branch ({len(hotel_branch_events)} events):")
            if hotel_branch_events:
                print("All hotel-related conversations are stored here:")
                for event in hotel_branch_events[-3:]:  # 최근 event 3개 표시
                    for payload in event.payload:
                        if "conversational" in payload:
                            role = payload["conversational"]["role"]
                            text = payload["conversational"]["content"]["text"]
                            print(f"  {role}: {text[:100]}...")
            else:
                print("  No events found - hotel assistant wasn't called yet")
        except Exception as e:
            print(f"  Hotel branch not created yet: {e}")

## 요약

이 Notebook에서는 병렬 실행이 가능한 강력한 Multi-Agent System을 구축하는 핵심 기능인 **AgentCore Memory Branching**을 구현했습니다.

### 핵심 요점:

1. **Memory Branching으로 병렬 실행 지원**
- 여러 Agent가 Memory 충돌 없이 동시에 실행
- 각 Agent가 branch를 통해 자체 대화 컨텍스트 유지
- Strands Agent Graph 및 기타 병렬 Agent framework에 필수

2. **컨텍스트 격리로 Agent 성능 개선**
- 전문 Agent가 간섭 없이 담당 도메인에 집중
- Agent 간 컨텍스트 오염 없음
- 더 깔끔하고 관련성 높은 대화

3. **격리된 컨텍스트가 있는 공유 세션**
- 단일 Memory 리소스 및 Session ID
- 서로 다른 Agent용 여러 branch
- 효율적인 리소스 활용


### 아키텍처 Pattern:

```
Memory Resource (memory_id)
  └── Session (session_id)
      ├── main branch → Coordinator Agent
      ├── flight_agent_memory → Flight Agent (can run in parallel)
      └── hotel_agent_memory → Hotel Agent (can run in parallel)
```

AgentCore Memory Branching을 사용하면 애플리케이션 요구 사항에 맞춰 확장 가능한 정교한 Multi-Agent System을 안전하고 효율적으로 구축할 수 있습니다.

## 리소스 정리
이 Notebook에서 사용한 리소스를 정리하기 위해 Memory를 삭제합니다.

In [ ]:
# client.delete_memory_and_wait(
#        memory_id = memory_id,
#        max_wait = 300,
#        poll_interval =10
# )